# 4. Hashing

In [ ]:
# --- BERBAGAI FUNGSI HASH ---
def hash_division(key, table_size):
    return key % table_size

def hash_mid_square(key, table_size):
    # Kuadratkan key, ambil digit tengah, modulo ukuran tabel
    squared = str(key ** 2)
    length = len(squared)
    if length > 2:
        mid = length // 2
        mid_digits = int(squared[mid - 1: mid + 1])
    else:
        mid_digits = int(squared)
    return mid_digits % table_size

def hash_folding(key, table_size):
    # Bagi key menjadi kelompok 2 digit, jumlahkan, modulo ukuran tabel
    key_str = str(key)
    parts = [int(key_str[i:i+2]) for i in range(0, len(key_str), 2)]
    return sum(parts) % table_size

def hash_string_polynomial(s, table_size):
    # Polynomial rolling hash untuk string
    p = 31
    hash_val = 0
    for char in s:
        hash_val = (hash_val * p + ord(char)) % table_size
    return hash_val

In [ ]:
# --- RESOLUSI KOLISI: LINEAR PROBING ---
def insert_linear_probing(table, key, value=None):
    size = len(table)
    idx = key % size
    collisions = 0
    
    for step in range(size):
        curr_idx = (idx + step) % size
        # Bisa menempati slot kosong (None) atau tanda "DELETED"
        if table[curr_idx] is None or table[curr_idx] == "DELETED":
            table[curr_idx] = key
            return curr_idx, collisions
        elif table[curr_idx] == key:
            # Update key jika sudah ada
            table[curr_idx] = key
            return curr_idx, collisions
        collisions += 1
        
    print(f"OVERFLOW! Tabel penuh dengan Linear Probing untuk key {key}")
    return -1, collisions

def search_linear_probing(table, key):
    size = len(table)
    idx = key % size
    
    for step in range(size):
        curr_idx = (idx + step) % size
        if table[curr_idx] is None:
            return -1 # Tidak ditemukan
        if table[curr_idx] == key:
            return curr_idx
    return -1

def delete_linear_probing(table, key):
    idx = search_linear_probing(table, key)
    if idx != -1:
        table[idx] = "DELETED"
        return True
    return False

In [ ]:
# --- RESOLUSI KOLISI: QUADRATIC PROBING ---
def insert_quadratic_probing(table, key):
    size = len(table)
    idx = key % size
    collisions = 0
    
    for i in range(size):
        curr_idx = (idx + i * i) % size
        if table[curr_idx] is None or table[curr_idx] == "DELETED":
            table[curr_idx] = key
            return curr_idx, collisions
        elif table[curr_idx] == key:
            return curr_idx, collisions
        collisions += 1
        
    print(f"Gagal menempatkan {key} dengan Quadratic Probing")
    return -1, collisions

def search_quadratic_probing(table, key):
    size = len(table)
    idx = key % size
    
    for i in range(size):
        curr_idx = (idx + i * i) % size
        if table[curr_idx] is None:
            return -1
        if table[curr_idx] == key:
            return curr_idx
    return -1

In [ ]:
# --- RESOLUSI KOLISI: DOUBLE HASHING ---
def secondary_hash(key):
    # Fungsi hash kedua (harus menghasilkan nilai > 0)
    return 7 - (key % 7)

def insert_double_hashing(table, key):
    size = len(table)
    idx = key % size
    h2 = secondary_hash(key)
    collisions = 0
    
    for step in range(size):
        curr_idx = (idx + step * h2) % size
        if table[curr_idx] is None or table[curr_idx] == "DELETED":
            table[curr_idx] = key
            return curr_idx, collisions
        elif table[curr_idx] == key:
            return curr_idx, collisions
        collisions += 1
        
    print(f"Gagal menempatkan {key} dengan Double Hashing")
    return -1, collisions

def search_double_hashing(table, key):
    size = len(table)
    idx = key % size
    h2 = secondary_hash(key)
    
    for step in range(size):
        curr_idx = (idx + step * h2) % size
        if table[curr_idx] is None:
            return -1
        if table[curr_idx] == key:
            return curr_idx
    return -1

In [ ]:
# --- RESOLUSI KOLISI: CHAINING ---
def insert_chaining(table, key):
    size = len(table)
    idx = key % size
    # Jika key belum ada di bucket list, tambahkan
    if key not in table[idx]:
        table[idx].append(key)
        return idx, len(table[idx]) - 1
    return idx, table[idx].index(key)

def search_chaining(table, key):
    size = len(table)
    idx = key % size
    if key in table[idx]:
        return idx, table[idx].index(key)
    return -1, -1

def delete_chaining(table, key):
    idx, pos = search_chaining(table, key)
    if idx != -1:
        table[idx].pop(pos)
        return True
    return False

In [ ]:
# --- HASH TABLE DENGAN DYNAMIC REHASHING ---
class SimpleHashTable:
    def __init__(self, capacity=5):
        self.capacity = capacity
        self.table = [None] * capacity
        self.count = 0
        
    def get_load_factor(self):
        return self.count / self.capacity
        
    def rehash(self):
        old_table = self.table
        old_capacity = self.capacity
        
        self.capacity = old_capacity * 2 + 1
        self.table = [None] * self.capacity
        self.count = 0
        print(f"[REHASHING] Load factor = {old_capacity/old_capacity:.2f} >= 0.75. Kapasitas diperbesar dari {old_capacity} ke {self.capacity}")
        
        for key in old_table:
            if key is not None and key != "DELETED":
                self.put(key)
                
    def put(self, key):
        if self.get_load_factor() >= 0.75:
            self.rehash()
            
        size = self.capacity
        idx = key % size
        for step in range(size):
            curr_idx = (idx + step) % size
            if self.table[curr_idx] is None or self.table[curr_idx] == "DELETED":
                self.table[curr_idx] = key
                self.count += 1
                return
                
    def __str__(self):
        return f"Table (LF={self.get_load_factor():.2f}): {self.table}"

In [ ]:
# --- VERIFIKASI & BENCHMARK DENGAN KASUS UJI ---
print("=== 1. VERIFIKASI FUNGSI HASH ===")
key_uji = 12345
size_tabel = 11
print(f"Key Uji: {key_uji} | Ukuran Tabel: {size_tabel}")
print(f"Division Method    : {hash_division(key_uji, size_tabel)}")
print(f"Mid-Square Method  : {hash_mid_square(key_uji, size_tabel)}")
print(f"Folding Method     : {hash_folding(key_uji, size_tabel)}")
print(f"String Polynomial  : {hash_string_polynomial('Strukdat', size_tabel)}")

print("\n=== 2. VERIFIKASI OPEN ADDRESSING & CHAINING ===")
data_uji = [54, 26, 93, 17, 77, 31, 44, 55, 20]
tabel_size = 11

# Uji Linear Probing
tabel_linear = [None] * tabel_size
linear_collisions = 0
for d in data_uji:
    _, coll = insert_linear_probing(tabel_linear, d)
    linear_collisions += coll
print(f"Hasil Linear Probing    : {tabel_linear} | Total Tabrakan: {linear_collisions}")

# Uji Quadratic Probing
tabel_quad = [None] * tabel_size
quad_collisions = 0
for d in data_uji:
    _, coll = insert_quadratic_probing(tabel_quad, d)
    quad_collisions += coll
print(f"Hasil Quadratic Probing : {tabel_quad} | Total Tabrakan: {quad_collisions}")

# Uji Double Hashing
tabel_double = [None] * tabel_size
double_collisions = 0
for d in data_uji:
    _, coll = insert_double_hashing(tabel_double, d)
    double_collisions += coll
print(f"Hasil Double Hashing    : {tabel_double} | Total Tabrakan: {double_collisions}")

# Uji Chaining
tabel_chain = [[] for _ in range(tabel_size)]
for d in data_uji:
    insert_chaining(tabel_chain, d)
print(f"Hasil Chaining          : {tabel_chain}")

print("\n=== 3. VERIFIKASI PENCARIAN & PENGHAPUSAN ===")
# Cari 17 di Linear Probing
idx_found = search_linear_probing(tabel_linear, 17)
print(f"Kunci 17 ditemukan di indeks Linear Probing: {idx_found}")
# Hapus 17
delete_linear_probing(tabel_linear, 17)
print(f"Linear Probing setelah kunci 17 dihapus    : {tabel_linear}")
print(f"Cari kembali kunci 17 (Hasil: {search_linear_probing(tabel_linear, 17)})")

print("\n=== 4. VERIFIKASI DYNAMIC REHASHING ===")
ht = SimpleHashTable(capacity=5)
print(ht)
ht.put(10)
ht.put(21)
ht.put(32)
print("Setelah memasukkan 3 elemen:")
print(ht)
ht.put(43) # Belum rehash (lf = 0.60, setelah masuk jadi 0.80)
print("Setelah elemen ke-4 masuk:")
print(ht)
ht.put(54) # Memicu rehash karena lf = 0.80 >= 0.75
print("Setelah elemen ke-5 dimasukkan (Rehashing Terpicu):")
print(ht)